In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [57]:
df = pd.read_csv(r"D:\OBD-II\obd_data_analysis\data\cleaned_dataset_v2.csv")

print("=== Basic Info ===")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nHead:")
display(df.head())

=== Basic Info ===
Shape: (249694, 14)
Columns: ['timestamp', 'trip_id', 'segment_id', 'row_in_segment', 'coolant_temp', 'map', 'rpm', 'speed', 'intake_temp', 'maf', 'tps', 'ambient_temp', 'accel_pedal_d', 'accel_pedal_e']

Head:


,timestamp,trip_id,segment_id,row_in_segment,coolant_temp,map,rpm,speed,intake_temp,maf,tps,ambient_temp,accel_pedal_d,accel_pedal_e
0,2017-07-05T07:16:30+02:00,trip_0001,trip_0001_seg_001,1,31.0,96.0,0.0,0.0,22.0,0.91,89.0,NaN,NaN,NaN
1,2017-07-05T07:16:31+02:00,trip_0001,trip_0001_seg_001,2,31.0,96.0,0.0,0.0,22.0,0.91,89.0,21.0,14.1,14.5
2,2017-07-05T07:16:32+02:00,trip_0001,trip_0001_seg_001,3,31.0,96.0,0.0,0.0,22.0,0.91,89.0,21.0,14.1,14.5
3,2017-07-05T07:16:33+02:00,trip_0001,trip_0001_seg_001,4,31.0,96.0,0.0,0.0,22.0,0.91,89.0,21.0,14.1,14.5
4,2017-07-05T07:16:34+02:00,trip_0001,trip_0001_seg_001,5,31.0,96.0,0.0,0.0,22.0,0.91,89.0,21.0,14.1,14.5


In [58]:
print("=== Dataset Overview ===")

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nUnique trips:", df['trip_id'].nunique())
print("Unique segments:", df['segment_id'].nunique())

print("\nRows per trip (top 10):")
display(df['trip_id'].value_counts().head(10))

=== Dataset Overview ===
Number of rows: 249694
Number of columns: 14

Unique trips: 81
Unique segments: 118

Rows per trip (top 10):


trip_id
trip_0079    7931
trip_0081    6523
trip_0023    6491
trip_0016    6242
trip_0070    5988
trip_0014    5187
trip_0025    5082
trip_0004    4973
trip_0027    4897
trip_0063    4839
Name: count, dtype: int64

In [59]:
print("=== Before parsing ===")
print(df['timestamp'].head())

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

print("\n=== After parsing ===")
print(df['timestamp'].head())

print("\nNull timestamps:", df['timestamp'].isna().sum())
print("Monotonic:", df['timestamp'].is_monotonic_increasing)
print("Range:", df['timestamp'].min(), "→", df['timestamp'].max())

=== Before parsing ===
0    2017-07-05T07:16:30+02:00
1    2017-07-05T07:16:31+02:00
2    2017-07-05T07:16:32+02:00
3    2017-07-05T07:16:33+02:00
4    2017-07-05T07:16:34+02:00
Name: timestamp, dtype: str

=== After parsing ===
0   2017-07-05 05:16:30+00:00
1   2017-07-05 05:16:31+00:00
2   2017-07-05 05:16:32+00:00
3   2017-07-05 05:16:33+00:00
4   2017-07-05 05:16:34+00:00
Name: timestamp, dtype: datetime64[us, UTC]

Null timestamps: 0
Monotonic: True
Range: 2017-07-05 05:16:30+00:00 → 2018-04-23 18:13:53+00:00


In [60]:
print("=== Data Types ===")
print(df.dtypes)

=== Data Types ===
timestamp         datetime64[us, UTC]
trip_id                           str
segment_id                        str
row_in_segment                  int64
coolant_temp                  float64
map                           float64
rpm                           float64
speed                         float64
intake_temp                   float64
maf                           float64
tps                           float64
ambient_temp                  float64
accel_pedal_d                 float64
accel_pedal_e                 float64
dtype: object


In [61]:
expected_cols = [
    'timestamp','trip_id','segment_id','row_in_segment',
    'coolant_temp','map','rpm','speed',
    'intake_temp','maf','tps','ambient_temp',
    'accel_pedal_d','accel_pedal_e'
]

missing_cols = set(expected_cols) - set(df.columns)
extra_cols = set(df.columns) - set(expected_cols)

print("Missing columns:", missing_cols)
print("Extra columns:", extra_cols)

dup_count = df.duplicated(
    subset=['trip_id','segment_id','row_in_segment']
).sum()

print("Duplicate rows:", dup_count)

def check_row_continuity(group):
    return group['row_in_segment'].is_monotonic_increasing

result = df.groupby(['trip_id','segment_id']).apply(check_row_continuity)

print("Segments with broken row order:", (~result).sum())

Missing columns: set()
Extra columns: set()
Duplicate rows: 0
Segments with broken row order: 0


In [62]:
df['dt'] = df.groupby(['trip_id','segment_id'])['timestamp'].diff().dt.total_seconds()

df['dt'].describe()
df['dt'].value_counts().head()

dt
1.0    249576
Name: count, dtype: int64

In [63]:
gaps = df[df['dt'] > 1.5]

print("Number of gaps:", len(gaps))
display(gaps.head())

Number of gaps: 0


,timestamp,trip_id,segment_id,row_in_segment,coolant_temp,map,rpm,speed,intake_temp,maf,tps,ambient_temp,accel_pedal_d,accel_pedal_e,dt


In [64]:
df.groupby(['trip_id','segment_id'])['timestamp'].apply(lambda x: x.diff().dt.total_seconds().max()).sort_values(ascending=False).head()

trip_id    segment_id       
trip_0001  trip_0001_seg_001    1.0
trip_0002  trip_0002_seg_001    1.0
trip_0003  trip_0003_seg_001    1.0
trip_0004  trip_0004_seg_001    1.0
trip_0005  trip_0005_seg_001    1.0
Name: timestamp, dtype: float64

In [65]:
missing_rate = df.isna().mean().sort_values(ascending=False)

print(missing_rate)

dt                0.000473
ambient_temp      0.000376
accel_pedal_e     0.000240
accel_pedal_d     0.000200
tps               0.000128
maf               0.000096
intake_temp       0.000076
speed             0.000048
rpm               0.000028
map               0.000012
coolant_temp      0.000004
row_in_segment    0.000000
timestamp         0.000000
trip_id           0.000000
segment_id        0.000000
dtype: float64


In [66]:
df['coolant_temp'].diff().abs().describe()

count    249691.000000
mean          0.057403
std           1.138715
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          86.000000
Name: coolant_temp, dtype: float64

In [67]:
df[df['coolant_temp'].diff().abs() > 30]

,timestamp,trip_id,segment_id,row_in_segment,coolant_temp,map,rpm,speed,intake_temp,maf,tps,ambient_temp,accel_pedal_d,accel_pedal_e,dt
3946,2017-07-05 17:17:19+00:00,trip_0002,trip_0002_seg_001,1,33.0,97.0,0.0,0.0,24.0,0.91,89.0,NaN,NaN,NaN,NaN
7269,2017-07-06 06:37:58+00:00,trip_0003,trip_0003_seg_001,1,30.0,98.0,0.0,0.0,20.0,0.91,89.0,21.0,14.1,14.5,NaN
8816,2017-07-06 16:54:28+00:00,trip_0004,trip_0004_seg_001,1,32.0,100.0,852.0,0.0,29.0,9.33,NaN,NaN,NaN,NaN,NaN
13789,2017-07-07 05:23:12+00:00,trip_0005,trip_0005_seg_001,1,32.0,96.0,0.0,0.0,23.0,0.91,89.0,22.0,14.1,14.5,NaN
20717,2017-07-11 06:24:29+00:00,trip_0008,trip_0008_seg_001,1,24.0,96.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225678,2018-03-23 16:24:03+00:00,trip_0076,trip_0076_seg_001,1,16.0,97.0,919.0,0.0,16.0,15.77,83.1,NaN,NaN,NaN,NaN
229417,2018-03-26 05:17:05+00:00,trip_0077,trip_0077_seg_001,1,9.0,96.0,0.0,0.0,2.0,0.86,89.0,6.0,14.1,14.5,NaN
234112,2018-03-29 14:45:10+00:00,trip_0079,trip_0079_seg_001,1,21.0,100.0,890.0,0.0,19.0,10.38,83.1,11.0,14.1,14.5,NaN
242043,2018-04-23 04:58:41+00:00,trip_0081,trip_0081_seg_001,1,17.0,96.0,0.0,0.0,11.0,0.88,89.0,16.0,14.1,14.5,NaN


In [68]:
z = (df['rpm'] - df['rpm'].mean()) / df['rpm'].std()
df[np.abs(z) > 3]

,timestamp,trip_id,segment_id,row_in_segment,coolant_temp,map,rpm,speed,intake_temp,maf,tps,ambient_temp,accel_pedal_d,accel_pedal_e,dt
4850,2017-07-05 17:32:23+00:00,trip_0002,trip_0002_seg_001,905,93.0,228.0,3108.0,184.0,34.0,108.77,83.5,29.0,78.8,78.0,1.0
4851,2017-07-05 17:32:24+00:00,trip_0002,trip_0002_seg_001,906,93.0,229.0,3115.0,185.0,34.0,108.47,83.5,29.0,78.8,78.0,1.0
4852,2017-07-05 17:32:25+00:00,trip_0002,trip_0002_seg_001,907,93.0,229.0,3137.0,186.0,34.0,109.74,83.5,29.0,78.8,78.0,1.0
4853,2017-07-05 17:32:26+00:00,trip_0002,trip_0002_seg_001,908,93.0,229.0,3145.0,186.0,34.0,109.55,83.5,29.0,78.8,78.0,1.0
4854,2017-07-05 17:32:27+00:00,trip_0002,trip_0002_seg_001,909,93.0,228.0,3149.0,187.0,34.0,109.94,83.5,29.0,78.8,78.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245584,2018-04-23 05:57:42+00:00,trip_0081,trip_0081_seg_001,3542,91.0,227.0,3254.0,193.0,23.0,113.88,83.5,18.0,78.0,77.6,1.0
245585,2018-04-23 05:57:43+00:00,trip_0081,trip_0081_seg_001,3543,91.0,227.0,3255.0,193.0,23.0,113.55,83.5,18.0,77.6,77.3,1.0
245586,2018-04-23 05:57:44+00:00,trip_0081,trip_0081_seg_001,3544,91.0,227.0,3250.0,193.0,23.0,113.66,83.5,18.0,78.0,77.3,1.0
245587,2018-04-23 05:57:45+00:00,trip_0081,trip_0081_seg_001,3545,91.0,146.0,3180.0,189.0,23.0,73.36,83.5,18.0,14.1,14.5,1.0


In [69]:
df[np.abs(z) > 3]['rpm'].describe()

count     637.000000
mean     3274.638932
std       140.651466
min      3105.000000
25%      3167.000000
50%      3244.000000
75%      3337.000000
max      4315.000000
Name: rpm, dtype: float64

In [70]:
df[np.abs(z) > 3].groupby('trip_id').size().sort_values(ascending=False)

trip_id
trip_0032    133
trip_0014     92
trip_0027     62
trip_0033     50
trip_0079     47
trip_0009     38
trip_0002     35
trip_0056     28
trip_0025     28
trip_0070     27
trip_0045     25
trip_0065     19
trip_0017     15
trip_0081     12
trip_0059     10
trip_0030      6
trip_0023      3
trip_0031      2
trip_0007      1
trip_0043      1
trip_0042      1
trip_0051      1
trip_0076      1
dtype: int64

In [71]:
df.groupby('trip_id')['speed'].mean()

trip_id
trip_0001    49.546629
trip_0002    78.112850
trip_0003    25.000646
trip_0004    75.057309
trip_0005    60.487496
               ...    
trip_0077    69.791186
trip_0078    55.705338
trip_0079    50.311688
trip_0080    32.185284
trip_0081    57.258163
Name: speed, Length: 81, dtype: float64